# Busca de uma árvore com ganho de informação

A primeira árvore agrupava as medidas em faixas de mesma largura antes de construir os ramos. Como havia valores extremos, muitas medidas diferentes caíam na mesma faixa. Aqui deixamos a árvore escolher os cortes numéricos pelo ganho de informação.

Continuamos com uma única árvore e a classe majoritária nas folhas. Usamos NumPy para acelerar as contagens e implementamos o treinamento e a previsão do zero.

As saídas estão salvas para leitura. A reexecução exige um kernel com NumPy.

### Resultado da busca

A grade inicial comparou 576 configurações. A seleção ampliada comparou outras 57.232 combinações de subconjuntos de atributos, estimativas de valores ausentes e limites de tamanho da árvore.

A candidata escolhida usa quatro medidas: **SYSTOLIC, GLUCOSE, HEMOGLOBIN e TRIGLYCERIDES**. O preenchimento é pela média do treinamento, a quantidade mínima por folha é 15 e a profundidade máxima é 6. As perguntas são cortes numéricos escolhidos pelo ganho de informação.

| Avaliação | Acurácia | F1 macro |
|---|---:|---:|
| Média das divisões usadas na seleção, sementes 42 e 73 | 90,39% | 90,37% |
| Conferência com semente 101 | 88,24% | 88,23% |
| 104 amostras reservadas | 84,62% | 84,61% |

Na reserva, foram 88 acertos. Acertamos 43 dos 50 casos de controle e 45 dos 54 casos de diabetes. A primeira árvore, com faixas fixas, acertou 66 dessas mesmas amostras.

Os 90,39% descrevem o resultado dentro da seleção. A conferência e a reserva mostram que esse valor ainda varia conforme as amostras. Esta candidata é mais forte que a primeira árvore, mas as três avaliações precisam ser apresentadas com seus nomes.

Depois de juntar caminhos que retornam a mesma classe, a árvore fica com seis perguntas, sete folhas e no máximo cinco perguntas por previsão. Essa forma compacta preserva os resultados da candidata.


### Referências e decisões desta implementação

| Parte | Referência nos PDFs |
|---|---|
| Árvore como sequência de testes nos atributos | Aula 5, páginas 16 e 17 |
| Busca de árvores compactas | Aula 5, página 18 |
| ID3, C4.5 e CART como exemplos de algoritmos | Aula 5, página 19 |
| Escolher o atributo pelo ganho de informação | Aula 5, páginas 20 e 23 a 28 |
| Classe majoritária e valor padrão | Aula 5, páginas 21 e 22 |
| Seleção de atributos | Aula 2, páginas 78 a 81 |
| Discretização supervisionada e binarização | Aula 2, páginas 86 a 91 |
| Avaliação e métricas | Aula 7, páginas 7 a 11 |

A árvore usa testes de dois resultados. Em um atributo numérico, o teste é “valor menor ou igual a um limite?”. Testamos os pontos intermediários entre valores consecutivos observados no nó e escolhemos o que dá maior ganho. Nas categorias, testamos uma categoria contra as demais.

O professor apresenta os conceitos acima. A varredura dos limites, a profundidade máxima e a quantidade mínima de exemplos por folha são escolhas desta implementação. Esta é uma adaptação com testes binários para medidas numéricas e categorias. O notebook anterior registra o ID3 categórico com as faixas prontas.

### Base e divisões

Repetimos a divisão anterior com semente 42. As 416 amostras de treinamento entram na escolha da configuração por validação cruzada em cinco partes. Depois avaliamos a mesma candidata com semente 73 e nas mesmas 104 amostras reservadas.

As 104 já foram consultadas em outros experimentos, então servem como comparação local. O conjunto de teste da competição fica reservado para receber as previsões de uma versão que decidirmos enviar.

In [1]:
from pathlib import Path
from itertools import product
from time import perf_counter
import csv
import hashlib
import json
import random
import sys

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho_base = raiz / "dados" / "treinamento_padronizado.csv"
with caminho_base.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

numericas = (
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC", "GLUCOSE",
    "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
)
categoricas = ("SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY", "GENDER", "BLOOD_GROUP", "RH_FACTOR")
colunas = numericas + categoricas
pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("A divisão exige um registro por paciente.")
classes = np.array([int(a["class"]) for a in amostras], dtype=np.int8)
if not np.all(np.isin(classes, [0, 1])):
    raise ValueError("Esperávamos as classes 0 e 1.")

controle = np.flatnonzero(classes == 0).tolist()
diabetes = np.flatnonzero(classes == 1).tolist()
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0, n1 = round(len(controle) * 0.20), round(len(diabetes) * 0.20)
reservadas = np.array(controle[:n0] + diabetes[:n1])
treinamento = np.array(controle[n0:] + diabetes[n1:])


def dividir(indices, semente):
    partes = [[] for _ in range(5)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if classes[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            partes[posicao % 5].append(i)
    for posicao, parte in enumerate(partes):
        referencia = [i for j, outra in enumerate(partes) if j != posicao for i in outra]
        yield np.array(referencia), np.array(parte)


print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")
print(f"Treinamento: {len(treinamento)} | Reservadas: {len(reservadas)}")
print("SHA-256:", hashlib.sha256(caminho_base.read_bytes()).hexdigest())

Python 3.12.14 | NumPy 2.3.5
Treinamento: 416 | Reservadas: 104
SHA-256: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032


### Valores ausentes e categorias

Mantemos a leitura numérica usada no projeto. Tratamos campos vazios, valores não finitos e medidas não positivas como ausentes. HEMOGLOBIN acima de 100 também fica ausente, sob a premissa anterior de HbA1c.

Comparamos preencher pela mediana ou pela média, sempre calculadas só com a referência da rodada. As categorias são representadas por códigos internos para facilitar as contagens. O teste categórico compara igualdade, então esses códigos não criam uma ordem entre categorias.

Os valores categóricos desconhecidos recebem uma categoria própria. Se uma categoria preenchida não tiver aparecido na referência, a previsão usa a maioria do nó em que encontrá-la. A árvore calcula cortes diretamente nas medidas, então a normalização não entra neste experimento.

In [2]:
def ler_numero(amostra, coluna):
    try:
        valor = float(amostra[coluna].strip().replace(",", "."))
    except ValueError:
        return np.nan
    if not np.isfinite(valor) or valor <= 0:
        return np.nan
    if coluna == "HEMOGLOBIN" and valor > 100:
        return np.nan
    return valor


def ler_categoria(amostra, coluna):
    valor = amostra[coluna].strip()
    if valor in {"", "desconhecido"} or (coluna == "GENDER" and valor == "N"):
        return "__desconhecido__"
    return valor


valores_numericos = np.array([[ler_numero(a, c) for c in numericas] for a in amostras])
valores_categoricos = {
    c: np.array([ler_categoria(a, c) for a in amostras]) for c in categoricas
}


def preparar(referencia, consulta, preenchimento):
    ref = np.zeros((len(referencia), len(colunas)))
    aval = np.zeros((len(consulta), len(colunas)))
    parametros = {}
    for j, coluna in enumerate(numericas):
        x = valores_numericos[referencia, j]
        y = valores_numericos[consulta, j]
        disponiveis = x[np.isfinite(x)]
        if not len(disponiveis):
            raise ValueError(f"Nenhuma medida disponível em {coluna}.")
        estimativa = np.median(disponiveis) if preenchimento == "mediana" else np.mean(disponiveis)
        ref[:, j] = np.where(np.isfinite(x), x, estimativa)
        aval[:, j] = np.where(np.isfinite(y), y, estimativa)
        parametros[coluna] = float(estimativa)

    mapas = {}
    for j, coluna in enumerate(categoricas, start=len(numericas)):
        x = valores_categoricos[coluna][referencia]
        y = valores_categoricos[coluna][consulta]
        mapa = {categoria: codigo for codigo, categoria in enumerate(sorted(set(x)))}
        ref[:, j] = [mapa[v] for v in x]
        aval[:, j] = [mapa.get(v, -1) for v in y]
        mapas[coluna] = mapa

    return ref, aval, parametros, mapas

### Entropia e ganho de informação

A entropia é zero quando um grupo tem uma só classe. Ela vale 1 quando metade é controle e metade diabetes. Para cada divisão possível, calculamos a média das entropias dos dois lados, ponderada pela quantidade de amostras.

O ganho é a entropia do grupo original menos essa média. A árvore escolhe o atributo e o corte com o maior ganho disponível. A fórmula de entropia e ganho aparece na aula 5, página 27.

In [3]:
def entropia(positivos, total):
    proporcao = np.asarray(positivos, dtype=float) / total
    p0 = 1 - proporcao
    parte1 = proporcao * np.log2(np.maximum(proporcao, 1e-15))
    parte0 = p0 * np.log2(np.maximum(p0, 1e-15))
    return -(parte1 + parte0)


def melhor_teste(x, y, atributos, minimo_folha):
    total = len(y)
    positivos = int(y.sum())
    antes = float(entropia(positivos, total))
    melhor = None
    melhor_ganho = 1e-12

    for j in atributos:
        valores = x[:, j]
        if j < len(numericas):
            ordem = np.argsort(valores, kind="stable")
            ordenados = valores[ordem]
            quantidades = np.arange(1, total)
            positivos_esquerda = np.cumsum(y[ordem])[:-1]
            candidatos = (
                (ordenados[:-1] < ordenados[1:])
                & (quantidades >= minimo_folha)
                & (total - quantidades >= minimo_folha)
            )
            if not candidatos.any():
                continue
            posicoes = np.flatnonzero(candidatos)
            n_esq = quantidades[posicoes]
            p_esq = positivos_esquerda[posicoes]
            depois = (
                n_esq * entropia(p_esq, n_esq)
                + (total - n_esq) * entropia(positivos - p_esq, total - n_esq)
            ) / total
            ganhos = antes - depois
            k = int(np.argmax(ganhos))
            ganho = float(ganhos[k])
            posicao = int(posicoes[k])
            limite = float((ordenados[posicao] + ordenados[posicao + 1]) / 2)
            tipo = "numero"
        else:
            unicos = np.unique(valores)
            ganhos = []
            limites = []
            for valor in unicos:
                esquerda = valores == valor
                n_esq = int(esquerda.sum())
                if n_esq < minimo_folha or total - n_esq < minimo_folha:
                    continue
                p_esq = int(y[esquerda].sum())
                depois = (
                    n_esq * entropia(p_esq, n_esq)
                    + (total - n_esq) * entropia(positivos - p_esq, total - n_esq)
                ) / total
                ganhos.append(float(antes - depois))
                limites.append(float(valor))
            if not ganhos:
                continue
            k = int(np.argmax(ganhos))
            ganho, limite = ganhos[k], limites[k]
            tipo = "categoria"

        if ganho > melhor_ganho:
            melhor_ganho = ganho
            melhor = (j, tipo, limite, ganho)
    return melhor

### Construção e tamanho da árvore

Cada nó guarda a classe majoritária do seu grupo. Se ele já for puro ou não houver uma divisão com ganho positivo que respeite o mínimo de exemplos por folha, criamos uma folha.

Também comparamos até onde deixar a árvore decidir. Profundidade 1 usa só a pergunta da raiz e duas folhas. Profundidade 2 permite mais uma pergunta em cada lado. Sem limite, seguimos enquanto houver uma divisão válida.

Para economizar contas, construímos a árvore inteira uma vez para cada referência, grupo de atributos e mínimo de exemplos por folha. Para avaliar uma profundidade menor, a previsão para no nó daquele nível e usa sua maioria. Isso produz as mesmas decisões que parar o crescimento nesse nível, pois as escolhas anteriores ao nível são iguais.

No empate entre classes, usamos a classe padrão do nó pai. Na raiz, o padrão é controle. Empates de ganho usam a ordem fixa dos atributos e dos limites.

In [4]:
def construir_arvore(x, y, atributos, minimo_folha, padrao=0):
    positivos = int(y.sum())
    if positivos * 2 == len(y):
        classe = padrao
    else:
        classe = int(positivos * 2 > len(y))
    no = {"classe": classe, "amostras": len(y), "positivos": positivos}
    if positivos == 0 or positivos == len(y) or len(y) < 2 * minimo_folha:
        return no

    teste = melhor_teste(x, y, atributos, minimo_folha)
    if teste is None:
        return no
    j, tipo, limite, ganho = teste
    esquerda = x[:, j] <= limite if tipo == "numero" else x[:, j] == limite
    no.update(atributo=j, tipo=tipo, limite=limite, ganho=ganho)
    if tipo == "categoria":
        no["conhecidas"] = np.unique(x[:, j]).tolist()
    no["sim"] = construir_arvore(x[esquerda], y[esquerda], atributos, minimo_folha, classe)
    no["nao"] = construir_arvore(x[~esquerda], y[~esquerda], atributos, minimo_folha, classe)
    return no


def prever(entrada, arvore, profundidade):
    no = arvore
    nivel = 0
    while "atributo" in no and (profundidade is None or nivel < profundidade):
        valor = entrada[no["atributo"]]
        if no["tipo"] == "categoria":
            if valor not in no["conhecidas"]:
                return no["classe"]
            caminho = "sim" if valor == no["limite"] else "nao"
        else:
            caminho = "sim" if valor <= no["limite"] else "nao"
        no = no[caminho]
        nivel += 1
    return no["classe"]


def contar(reais, previstas):
    return np.array([
        np.sum((reais == 0) & (previstas == 0)),
        np.sum((reais == 0) & (previstas == 1)),
        np.sum((reais == 1) & (previstas == 0)),
        np.sum((reais == 1) & (previstas == 1)),
    ])


def razao(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(np.asarray(contagens), -1, 0)
    return {
        "acuracia": razao(tn + tp, tn + fp + fn + tp),
        "f1": (razao(2 * tn, 2 * tn + fp + fn) + razao(2 * tp, 2 * tp + fp + fn)) / 2,
        "precisao0": razao(tn, tn + fn),
        "precisao1": razao(tp, tp + fp),
        "precisao": (razao(tn, tn + fn) + razao(tp, tp + fp)) / 2,
        "recall0": razao(tn, tn + fp),
        "recall1": razao(tp, tp + fn),
    }

### Configurações da busca

Comparamos seis grupos de atributos, dois preenchimentos, seis tamanhos mínimos de folha e oito limites de profundidade. A escolha é feita pelo maior F1 macro médio nas cinco rodadas com semente 42.

Em empate exato de F1, preferimos menor profundidade, menos atributos, maior mínimo de exemplos por folha e a ordem fixa da grade. Esses critérios de empate favorecem uma explicação mais curta quando os resultados são iguais.

In [5]:
grupos = {
    "Dois exames": ("GLUCOSE", "HEMOGLOBIN"),
    "Três exames": ("GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"),
    "Nove medidas": numericas,
    "Exames e hábitos": ("GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES", "SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY"),
    "Medidas e histórico": numericas + ("FAMILY_HISTORY",),
    "Todos os quinze": colunas,
}
preenchimentos = ("mediana", "media")
minimos_folha = (1, 2, 3, 5, 10, 20)
profundidades = (1, 2, 3, 4, 5, 6, 8, None)
configuracoes = list(product(grupos, preenchimentos, minimos_folha, profundidades))
indices_grupos = {
    nome: [colunas.index(c) for c in atributos] for nome, atributos in grupos.items()
}


def avaliar_grade(referencia, consulta):
    resultados = []
    for preenchimento in preenchimentos:
        x, aval, parametros, mapas = preparar(referencia, consulta, preenchimento)
        for nome in grupos:
            for minimo in minimos_folha:
                arvore = construir_arvore(x, classes[referencia], indices_grupos[nome], minimo)
                for profundidade in profundidades:
                    previstas = np.array([prever(entrada, arvore, profundidade) for entrada in aval])
                    resultados.append(((nome, preenchimento, minimo, profundidade), contar(classes[consulta], previstas)))
    por_configuracao = dict(resultados)
    return np.array([por_configuracao[c] for c in configuracoes])


def chave_desempate(indice, f1):
    nome, preenchimento, minimo, profundidade = configuracoes[indice]
    return (-f1[indice], profundidade if profundidade is not None else 999, len(grupos[nome]), -minimo, indice)


inicio = perf_counter()
rodadas_busca = []
for numero, (referencia, consulta) in enumerate(dividir(treinamento, 42), start=1):
    rodadas_busca.append(avaliar_grade(referencia, consulta))
    print(f"Rodada {numero} concluída")

rodadas_busca = np.array(rodadas_busca)
f1_busca = metricas(rodadas_busca)["f1"].mean(axis=0)
acc_busca = metricas(rodadas_busca)["acuracia"].mean(axis=0)
ordem = sorted(range(len(configuracoes)), key=lambda i: chave_desempate(i, f1_busca))
escolhida = ordem[0]
print(f"Configurações: {len(configuracoes)}")
print(f"Tempo da busca: {perf_counter() - inicio:.1f} s")

Rodada 1 concluída
Rodada 2 concluída
Rodada 3 concluída
Rodada 4 concluída
Rodada 5 concluída
Configurações: 576
Tempo da busca: 3.4 s


In [6]:
print("As dez primeiras configurações na busca")
for indice in ordem[:10]:
    nome, preenchimento, minimo, profundidade = configuracoes[indice]
    print(
        f"Acc {acc_busca[indice]:.2%} | F1 {f1_busca[indice]:.2%} | "
        f"{nome} | {preenchimento} | Mínimo folha {minimo} | Profundidade {profundidade}"
    )
print()
print("Escolhida:", configuracoes[escolhida])

As dez primeiras configurações na busca
Acc 89.66% | F1 89.65% | Nove medidas | media | Mínimo folha 10 | Profundidade 6
Acc 89.42% | F1 89.40% | Medidas e histórico | media | Mínimo folha 10 | Profundidade 6
Acc 89.42% | F1 89.40% | Todos os quinze | media | Mínimo folha 10 | Profundidade 6
Acc 89.42% | F1 89.40% | Nove medidas | media | Mínimo folha 10 | Profundidade 8
Acc 89.42% | F1 89.40% | Nove medidas | media | Mínimo folha 10 | Profundidade None
Acc 89.18% | F1 89.16% | Medidas e histórico | media | Mínimo folha 5 | Profundidade 6
Acc 89.18% | F1 89.16% | Todos os quinze | media | Mínimo folha 5 | Profundidade 6
Acc 89.18% | F1 89.15% | Medidas e histórico | media | Mínimo folha 10 | Profundidade 8
Acc 89.18% | F1 89.15% | Todos os quinze | media | Mínimo folha 10 | Profundidade 8
Acc 89.18% | F1 89.15% | Medidas e histórico | media | Mínimo folha 10 | Profundidade None

Escolhida: ('Nove medidas', 'media', 10, 6)


### Conferência com outra semente e nas amostras reservadas

Fixamos a candidata escolhida na semente 42 e repetimos a validação cruzada com semente 73. Como são as mesmas 416 pessoas em outra divisão, essa repetição mostra a sensibilidade ao embaralhamento.

Depois treinamos a candidata com as 416 e avaliamos nas mesmas 104 amostras reservadas. Comparamos com as matrizes já registradas dos modelos anteriores nessa divisão. A comparação com o primeiro modelo de árvore ajuda a medir a mudança de representação.

In [7]:
def ajustar_candidata(referencia, consulta, configuracao):
    nome, preenchimento, minimo, profundidade = configuracao
    x, aval, parametros, mapas = preparar(referencia, consulta, preenchimento)
    arvore = construir_arvore(x, classes[referencia], indices_grupos[nome], minimo)
    previstas = np.array([prever(entrada, arvore, profundidade) for entrada in aval])
    return contar(classes[consulta], previstas), arvore, parametros, mapas


repeticao = np.array([
    ajustar_candidata(ref, aval, configuracoes[escolhida])[0]
    for ref, aval in dividir(treinamento, 73)
])
reserva, arvore_final, parametros_finais, mapas_finais = ajustar_candidata(
    treinamento, reservadas, configuracoes[escolhida]
)


def mostrar(nome, contagens):
    m = metricas(contagens)
    valores = [np.mean(m[c]) for c in ("acuracia", "f1", "precisao", "recall0", "recall1")]
    print(f"{nome:23} | " + " | ".join(f"{v:>8.2%}" for v in valores))


print("Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
mostrar("Busca, semente 42", rodadas_busca[:, escolhida])
mostrar("Repetição, semente 73", repeticao)
mostrar("Reserva de 104", reserva)

print()
print("Comparação nas 104 amostras reservadas")
mostrar("Primeira árvore", np.array([28, 22, 16, 38]))
mostrar("KNN seleção de aula", np.array([41, 9, 7, 47]))
mostrar("Primeiro KNN enviado", np.array([44, 6, 8, 46]))
mostrar("Árvore candidata", reserva)

print("Matriz da candidata [controle, diabetes]:", reserva.reshape(2, 2).tolist())
print(f"Acertos: {reserva[0] + reserva[3]} de {len(reservadas)}")
print(f"Precisão controle: {metricas(reserva)['precisao0']:.2%}")
print(f"Precisão diabetes: {metricas(reserva)['precisao1']:.2%}")

Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Busca, semente 42       |   89.66% |   89.65% |   89.85% |   91.00% |   88.40%
Repetição, semente 73   |   89.90% |   89.88% |   90.18% |   90.00% |   89.84%
Reserva de 104          |   83.65% |   83.58% |   83.78% |   80.00% |   87.04%

Comparação nas 104 amostras reservadas
Primeira árvore         |   63.46% |   63.12% |   63.48% |   56.00% |   70.37%
KNN seleção de aula     |   84.62% |   84.56% |   84.67% |   82.00% |   87.04%
Primeiro KNN enviado    |   86.54% |   86.53% |   86.54% |   88.00% |   85.19%
Árvore candidata        |   83.65% |   83.58% |   83.78% |   80.00% |   87.04%
Matriz da candidata [controle, diabetes]: [[40, 10], [7, 47]]
Acertos: 87 de 104
Precisão controle: 85.11%
Precisão diabetes: 82.46%


### Como explicar a candidata

Cada pergunta leva a um dos dois caminhos. Uma folha retorna a classe mais frequente entre os exemplos que chegaram a ela. O código mostra os testes usados pela candidata até o limite de profundidade escolhido, com os nomes originais dos atributos e das categorias.

Os cortes são encontrados nos dados de treinamento e não representam limites clínicos definidos pelo professor.

In [8]:
def mostrar_arvore(no, limite, mapas, nivel=0):
    recuo = "  " * nivel
    if "atributo" not in no or (limite is not None and nivel >= limite):
        print(f"{recuo}Prever classe {no['classe']} | Exemplos {no['amostras']}")
        return
    coluna = colunas[no["atributo"]]
    if no["tipo"] == "numero":
        pergunta = f"{coluna} <= {no['limite']:.3f}"
    else:
        reverso = {codigo: categoria for categoria, codigo in mapas[coluna].items()}
        pergunta = f"{coluna} = {reverso[no['limite']]}"
    print(f"{recuo}{pergunta}? | Ganho {no['ganho']:.3f} | Exemplos {no['amostras']}")
    print(f"{recuo}Sim:")
    mostrar_arvore(no["sim"], limite, mapas, nivel + 1)
    print(f"{recuo}Não:")
    mostrar_arvore(no["nao"], limite, mapas, nivel + 1)


limite_final = configuracoes[escolhida][3]
limite_exibicao = min(limite_final, 4) if limite_final is not None else 4
print("Primeiros níveis da árvore candidata")
mostrar_arvore(arvore_final, limite_exibicao, mapas_finais)

Primeiros níveis da árvore candidata
HEMOGLOBIN <= 6.350? | Ganho 0.253 | Exemplos 416
Sim:
  GLUCOSE <= 129.500? | Ganho 0.304 | Exemplos 157
  Sim:
    HEMOGLOBIN <= 5.450? | Ganho 0.065 | Exemplos 138
    Sim:
      Prever classe 0 | Exemplos 79
    Não:
      TRIGLYCERIDES <= 112.500? | Ganho 0.092 | Exemplos 59
      Sim:
        Prever classe 0 | Exemplos 23
      Não:
        Prever classe 0 | Exemplos 36
  Não:
    Prever classe 1 | Exemplos 19
Não:
  HEMOGLOBIN <= 11.000? | Ganho 0.210 | Exemplos 259
  Sim:
    HEMOGLOBIN <= 6.450? | Ganho 0.064 | Exemplos 113
    Sim:
      Prever classe 1 | Exemplos 10
    Não:
      Prever classe 1 | Exemplos 103
  Não:
    GLUCOSE <= 120.000? | Ganho 0.256 | Exemplos 146
    Sim:
      SYSTOLIC <= 119.500? | Ganho 0.253 | Exemplos 110
      Sim:
        Prever classe 0 | Exemplos 43
      Não:
        Prever classe 1 | Exemplos 67
    Não:
      Prever classe 1 | Exemplos 36


### Registro completo e consulta

Guardamos as configurações e as contagens de cada rodada para recuperar os resultados sem repetir a busca. As contagens seguem TN, FP, FN e TP, com controle como classe 0.

A função de consulta mostra uma configuração e suas matrizes por rodada. O exemplo consulta a primeira colocada.

In [9]:
caminho_registro = raiz / "resultados" / "arvore_busca_aulas.npz"
np.savez_compressed(
    caminho_registro,
    catalogo=json.dumps({
        "configuracoes": configuracoes, "grupos": grupos,
        "semente_busca": 42, "semente_repeticao": 73,
        "sha256_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest(),
    }),
    busca=rodadas_busca, repeticao=repeticao, reserva=reserva, escolhida=escolhida,
)


def consultar(indice):
    if not 0 <= indice < len(configuracoes):
        raise ValueError("Índice fora da grade.")
    print("Configuração:", configuracoes[indice])
    mostrar("Média das rodadas", rodadas_busca[:, indice])
    for numero, matriz in enumerate(rodadas_busca[:, indice], start=1):
        print(f"Rodada {numero}: {matriz.reshape(2, 2).tolist()}")


print("Registro:", caminho_registro.relative_to(raiz))
consultar(escolhida)

Registro: resultados\arvore_busca_aulas.npz
Configuração: ('Nove medidas', 'media', 10, 6)
Média das rodadas       |   89.66% |   89.65% |   89.85% |   91.00% |   88.40%
Rodada 1: [[34, 6], [3, 41]]
Rodada 2: [[37, 3], [6, 37]]
Rodada 3: [[38, 2], [6, 37]]
Rodada 4: [[36, 4], [8, 35]]
Rodada 5: [[37, 3], [2, 41]]


### Seleção ampliada dos atributos numéricos

Na primeira grade, a candidata teve resultados próximos de 90% nas cinco divisões, mas ficou abaixo disso nas 104 amostras reservadas. Agora comparamos os 511 subconjuntos das nove medidas numéricas. A seleção de atributos está na aula 2, páginas 78 a 81.

Para cada subconjunto, testamos média e mediana, sete quantidades mínimas de exemplos por folha e oito profundidades. Usamos as mesmas duas sementes, 42 e 73, para escolher pelo F1 macro médio das dez rodadas. A repetição usa as mesmas pessoas e serve para medir a variação entre partições.

Em empate exato de F1 médio, preferimos o maior dos resultados mínimos entre as duas sementes. Depois preferimos menos atributos, menor profundidade e maior mínimo por folha. Conferimos a configuração escolhida uma vez com a semente 101, definida antes dessa conferência.

In [10]:
from itertools import combinations

subconjuntos_numericos = [
    grupo for quantidade in range(1, len(numericas) + 1)
    for grupo in combinations(range(len(numericas)), quantidade)
]
folhas_selecao = (3, 5, 8, 10, 12, 15, 20)
profundidades_selecao = (1, 2, 3, 4, 5, 6, 8, None)
configuracoes_selecao = list(product(
    range(len(subconjuntos_numericos)), preenchimentos, folhas_selecao, profundidades_selecao
))
rodadas_selecao = np.zeros((10, len(configuracoes_selecao), 4), dtype=np.int16)

inicio = perf_counter()
rodada = 0
for semente in (42, 73):
    for numero, (referencia, consulta) in enumerate(dividir(treinamento, semente), start=1):
        preparados = {
            preenchimento: preparar(referencia, consulta, preenchimento)[:2]
            for preenchimento in preenchimentos
        }
        posicao = 0
        for grupo in subconjuntos_numericos:
            for preenchimento in preenchimentos:
                x, aval = preparados[preenchimento]
                for minimo in folhas_selecao:
                    arvore = construir_arvore(x, classes[referencia], grupo, minimo)
                    for profundidade in profundidades_selecao:
                        previstas = np.array([prever(a, arvore, profundidade) for a in aval])
                        rodadas_selecao[rodada, posicao] = contar(classes[consulta], previstas)
                        posicao += 1
        print(f"Semente {semente}, rodada {numero} concluída", flush=True)
        rodada += 1

m_selecao = metricas(rodadas_selecao)
medias_f1_selecao = m_selecao["f1"].mean(axis=0)
medias_acc_selecao = m_selecao["acuracia"].mean(axis=0)
f1_42 = m_selecao["f1"][:5].mean(axis=0)
f1_73 = m_selecao["f1"][5:].mean(axis=0)


def chave_selecao(i):
    grupo, preenchimento, minimo, profundidade = configuracoes_selecao[i]
    return (
        -medias_f1_selecao[i], -min(f1_42[i], f1_73[i]),
        len(subconjuntos_numericos[grupo]),
        profundidade if profundidade is not None else 999,
        -minimo, i,
    )


ordem_selecao = sorted(range(len(configuracoes_selecao)), key=chave_selecao)
escolhida_selecao = ordem_selecao[0]
print(f"Combinações da seleção ampliada: {len(configuracoes_selecao)}")
print(f"Tempo da seleção ampliada: {perf_counter() - inicio:.1f} s")

Semente 42, rodada 1 concluída
Semente 42, rodada 2 concluída
Semente 42, rodada 3 concluída
Semente 42, rodada 4 concluída
Semente 42, rodada 5 concluída
Semente 73, rodada 1 concluída
Semente 73, rodada 2 concluída
Semente 73, rodada 3 concluída
Semente 73, rodada 4 concluída
Semente 73, rodada 5 concluída
Combinações da seleção ampliada: 57232
Tempo da seleção ampliada: 359.1 s


In [11]:
def descrever_selecao(i):
    grupo, preenchimento, minimo, profundidade = configuracoes_selecao[i]
    atributos = [colunas[j] for j in subconjuntos_numericos[grupo]]
    return f"{atributos} | {preenchimento} | Folha {minimo} | Profundidade {profundidade}"


for i in ordem_selecao[:10]:
    print(
        f"Acc {medias_acc_selecao[i]:.2%} | F1 {medias_f1_selecao[i]:.2%} | "
        f"F1 s42 {f1_42[i]:.2%} | F1 s73 {f1_73[i]:.2%} | {descrever_selecao(i)}"
    )

Acc 90.39% | F1 90.37% | F1 s42 89.41% | F1 s73 91.34% | ['SYSTOLIC', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES'] | media | Folha 15 | Profundidade 6
Acc 90.39% | F1 90.37% | F1 s42 89.41% | F1 s73 91.34% | ['SYSTOLIC', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES'] | media | Folha 15 | Profundidade 8
Acc 90.39% | F1 90.37% | F1 s42 89.41% | F1 s73 91.34% | ['SYSTOLIC', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES'] | media | Folha 15 | Profundidade None
Acc 90.39% | F1 90.37% | F1 s42 89.41% | F1 s73 91.34% | ['HEIGHT', 'SYSTOLIC', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES'] | media | Folha 15 | Profundidade 6
Acc 90.39% | F1 90.37% | F1 s42 89.41% | F1 s73 91.34% | ['SYSTOLIC', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES', 'SODIUM'] | media | Folha 15 | Profundidade 6
Acc 90.39% | F1 90.37% | F1 s42 89.41% | F1 s73 91.34% | ['HEIGHT', 'SYSTOLIC', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES'] | media | Folha 15 | Profundidade 8
Acc 90.39% | F1 90.37% | F1 s42 89.41% | F1 s73 91.34% | ['SYSTOLIC', 'GLUCOSE'

#### Conferência da candidata escolhida

Fixamos a configuração escolhida pelas sementes 42 e 73 e avaliamos com cinco partes da semente 101. Depois fazemos a comparação local nas 104 amostras reservadas, sem usar o resultado para trocar a candidata desta busca.

A descrição da árvore continua sendo a mesma, com perguntas binárias escolhidas pelo ganho de informação e classe majoritária nas folhas.

In [12]:
grupo_escolhido, preenchimento_escolhido, folha_escolhida, profundidade_escolhida = (
    configuracoes_selecao[escolhida_selecao]
)
atributos_escolhidos = subconjuntos_numericos[grupo_escolhido]


def ajustar_selecao(referencia, consulta):
    x, aval, parametros, mapas = preparar(referencia, consulta, preenchimento_escolhido)
    arvore = construir_arvore(x, classes[referencia], atributos_escolhidos, folha_escolhida)
    previstas = np.array([prever(a, arvore, profundidade_escolhida) for a in aval])
    return contar(classes[consulta], previstas), arvore, parametros, mapas


conferencia_101 = np.array([
    ajustar_selecao(ref, aval)[0] for ref, aval in dividir(treinamento, 101)
])
reserva_selecao, arvore_selecao, parametros_selecao, mapas_selecao = ajustar_selecao(
    treinamento, reservadas
)

print("Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
mostrar("Seleção, sementes 42/73", rodadas_selecao[:, escolhida_selecao])
mostrar("Conferência, semente 101", conferencia_101)
mostrar("Reserva, 104 amostras", reserva_selecao)
print("Matriz da candidata:", reserva_selecao.reshape(2, 2).tolist())
print("Acertos:", int(reserva_selecao[0] + reserva_selecao[3]), "de", len(reservadas))
print("Configuração:", descrever_selecao(escolhida_selecao))

Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Seleção, sementes 42/73 |   90.39% |   90.37% |   90.51% |   91.50% |   89.36%
Conferência, semente 101 |   88.24% |   88.23% |   88.36% |   90.00% |   86.64%
Reserva, 104 amostras   |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Matriz da candidata: [[43, 7], [9, 45]]
Acertos: 88 de 104
Configuração: ['SYSTOLIC', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES'] | media | Folha 15 | Profundidade 6


In [13]:
with np.load(caminho_registro, allow_pickle=False) as registro:
    dados = {chave: registro[chave].copy() for chave in registro.files}
dados["selecao_subconjuntos"] = json.dumps(subconjuntos_numericos)
dados["selecao_configuracoes"] = json.dumps(configuracoes_selecao)
dados["selecao_busca"] = rodadas_selecao
dados["selecao_escolhida"] = escolhida_selecao
dados["selecao_conferencia_101"] = conferencia_101
dados["selecao_reserva"] = reserva_selecao
np.savez_compressed(caminho_registro, **dados)
print("Seleção ampliada registrada:", caminho_registro.relative_to(raiz))

Seleção ampliada registrada: resultados\arvore_busca_aulas.npz


### Perguntas da árvore final

A saída abaixo mostra a árvore escolhida na seleção ampliada. Cada caminho começa na primeira pergunta e termina na previsão de uma classe. As respostas são encadeadas até chegar à folha.

Os limites foram aprendidos com as 416 amostras de treinamento. Eles são decisões do modelo nesta base e devem ser apresentados como tais.


In [14]:
def tamanho_arvore(no, profundidade, nivel=0):
    if "atributo" not in no or (profundidade is not None and nivel >= profundidade):
        return 0, 1, nivel
    internos_sim, folhas_sim, altura_sim = tamanho_arvore(no["sim"], profundidade, nivel + 1)
    internos_nao, folhas_nao, altura_nao = tamanho_arvore(no["nao"], profundidade, nivel + 1)
    return 1 + internos_sim + internos_nao, folhas_sim + folhas_nao, max(altura_sim, altura_nao)


perguntas, folhas, maior_caminho = tamanho_arvore(arvore_selecao, profundidade_escolhida)
print(f"Perguntas: {perguntas} | Folhas: {folhas} | Maior caminho: {maior_caminho} perguntas")
mostrar_arvore(arvore_selecao, profundidade_escolhida, mapas_selecao)


Perguntas: 12 | Folhas: 13 | Maior caminho: 6 perguntas
HEMOGLOBIN <= 6.350? | Ganho 0.253 | Exemplos 416
Sim:
  GLUCOSE <= 129.500? | Ganho 0.304 | Exemplos 157
  Sim:
    HEMOGLOBIN <= 5.450? | Ganho 0.065 | Exemplos 138
    Sim:
      Prever classe 0 | Exemplos 79
    Não:
      TRIGLYCERIDES <= 112.500? | Ganho 0.092 | Exemplos 59
      Sim:
        Prever classe 0 | Exemplos 23
      Não:
        SYSTOLIC <= 125.500? | Ganho 0.042 | Exemplos 36
        Sim:
          Prever classe 0 | Exemplos 19
        Não:
          Prever classe 0 | Exemplos 17
  Não:
    Prever classe 1 | Exemplos 19
Não:
  HEMOGLOBIN <= 11.000? | Ganho 0.210 | Exemplos 259
  Sim:
    SYSTOLIC <= 113.500? | Ganho 0.051 | Exemplos 113
    Sim:
      Prever classe 1 | Exemplos 16
    Não:
      Prever classe 1 | Exemplos 97
  Não:
    GLUCOSE <= 120.000? | Ganho 0.256 | Exemplos 146
    Sim:
      SYSTOLIC <= 119.500? | Ganho 0.253 | Exemplos 110
      Sim:
        GLUCOSE <= 100.500? | Ganho 0.087 | Exemplos 4

#### Versão compacta para explicar

Se os dois caminhos de uma pergunta terminam na mesma classe, podemos juntar os dois em uma folha. Essa simplificação lógica preserva as previsões. Ela ajuda a apresentar a árvore de forma mais curta e se relaciona à busca por árvores compactas discutida na aula 5, página 18. A regra específica de juntar os ramos iguais é uma escolha desta implementação.

A saída mostra todos os testes da árvore compacta. Os limites são exibidos com três casas decimais para leitura, e o cálculo da previsão usa os valores originais armazenados.


In [15]:
def simplificar_arvore(no, profundidade, nivel=0):
    folha = {
        "classe": no["classe"],
        "amostras": no["amostras"],
        "positivos": no["positivos"],
    }
    if "atributo" not in no or (profundidade is not None and nivel >= profundidade):
        return folha

    copia = no.copy()
    copia["sim"] = simplificar_arvore(no["sim"], profundidade, nivel + 1)
    copia["nao"] = simplificar_arvore(no["nao"], profundidade, nivel + 1)
    sim, nao = copia["sim"], copia["nao"]

    if "atributo" not in sim and "atributo" not in nao and sim["classe"] == nao["classe"]:
        if no["tipo"] == "numero" or sim["classe"] == no["classe"]:
            folha["classe"] = sim["classe"]
            return folha
    return copia


arvore_simplificada = simplificar_arvore(arvore_selecao, profundidade_escolhida)
perguntas_compacta, folhas_compacta, maior_caminho_compacta = tamanho_arvore(arvore_simplificada, None)
print(
    f"Perguntas: {perguntas_compacta} | Folhas: {folhas_compacta} | "
    f"Maior caminho: {maior_caminho_compacta} perguntas"
)
mostrar_arvore(arvore_simplificada, None, mapas_selecao)


Perguntas: 6 | Folhas: 7 | Maior caminho: 5 perguntas
HEMOGLOBIN <= 6.350? | Ganho 0.253 | Exemplos 416
Sim:
  GLUCOSE <= 129.500? | Ganho 0.304 | Exemplos 157
  Sim:
    Prever classe 0 | Exemplos 138
  Não:
    Prever classe 1 | Exemplos 19
Não:
  HEMOGLOBIN <= 11.000? | Ganho 0.210 | Exemplos 259
  Sim:
    Prever classe 1 | Exemplos 113
  Não:
    GLUCOSE <= 120.000? | Ganho 0.256 | Exemplos 146
    Sim:
      SYSTOLIC <= 119.500? | Ganho 0.253 | Exemplos 110
      Sim:
        Prever classe 0 | Exemplos 43
      Não:
        TRIGLYCERIDES <= 142.000? | Ganho 0.131 | Exemplos 67
        Sim:
          Prever classe 0 | Exemplos 25
        Não:
          Prever classe 1 | Exemplos 42
    Não:
      Prever classe 1 | Exemplos 36


### Acrescentar tabagismo, atividade física e histórico familiar

Acrescentamos SMOKER, PHYSICAL_ACTIVITY e FAMILY_HISTORY às quatro medidas da candidata. Mantemos os cortes por ganho de informação, preenchimento pela média, mínimo de 15 exemplos por folha e profundidade máxima 6.

As categorias entram como perguntas de igualdade, por exemplo “SMOKER é ex_fumante?”. Elas participam da escolha pelo mesmo ganho de informação dos atributos numéricos. Uma categoria desconhecida é representada como categoria própria, e uma categoria inédita na referência usa a maioria do nó.

A comparação usa as mesmas cinco partes nas sementes 42, 73 e 101. A média de 42 e 73 permite comparar com a seleção anterior. A semente 101 e as 104 amostras reservadas mostram como a mudança se comporta nas outras avaliações locais já usadas no projeto.

In [16]:
novas_categorias = ("SMOKER", "PHYSICAL_ACTIVITY", "FAMILY_HISTORY")
indices_categorias = tuple(sorted(colunas.index(c) for c in novas_categorias))
variantes_habitos = {
    "Quatro medidas": tuple(atributos_escolhidos),
    "Medidas e hábitos": tuple(atributos_escolhidos) + indices_categorias,
}
resultados_habitos = {}

for semente in (42, 73, 101):
    por_variante = {nome: [] for nome in variantes_habitos}
    for referencia, consulta in dividir(treinamento, semente):
        x, aval, parametros, mapas = preparar(referencia, consulta, preenchimento_escolhido)
        for nome, atributos in variantes_habitos.items():
            arvore = construir_arvore(x, classes[referencia], atributos, folha_escolhida)
            previstas = np.array([
                prever(entrada, arvore, profundidade_escolhida) for entrada in aval
            ])
            por_variante[nome].append(contar(classes[consulta], previstas))
    for nome, rodadas in por_variante.items():
        resultados_habitos[semente, nome] = np.array(rodadas)

print("Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for nome in variantes_habitos:
    rodadas = np.concatenate((
        resultados_habitos[42, nome], resultados_habitos[73, nome]
    ))
    mostrar(f"{nome}, s42/73", rodadas)
for nome in variantes_habitos:
    mostrar(f"{nome}, s101", resultados_habitos[101, nome])

Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Quatro medidas, s42/73  |   90.39% |   90.37% |   90.51% |   91.50% |   89.36%
Medidas e hábitos, s42/73 |   90.39% |   90.37% |   90.51% |   91.50% |   89.36%
Quatro medidas, s101    |   88.24% |   88.23% |   88.36% |   90.00% |   86.64%
Medidas e hábitos, s101 |   88.24% |   88.23% |   88.36% |   90.00% |   86.64%


In [17]:
x, aval, parametros_habitos, mapas_habitos = preparar(
    treinamento, reservadas, preenchimento_escolhido
)
reservas_habitos = {}
arvores_habitos = {}
previsoes_habitos = {}

for nome, atributos in variantes_habitos.items():
    arvore = construir_arvore(x, classes[treinamento], atributos, folha_escolhida)
    previstas = np.array([prever(a, arvore, profundidade_escolhida) for a in aval])
    arvores_habitos[nome] = arvore
    previsoes_habitos[nome] = previstas
    reservas_habitos[nome] = contar(classes[reservadas], previstas)

print("Nas 104 amostras reservadas")
print("Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for nome, contagens in reservas_habitos.items():
    mostrar(nome, contagens)
    print("Matriz:", contagens.reshape(2, 2).tolist())

antes = previsoes_habitos["Quatro medidas"]
depois = previsoes_habitos["Medidas e hábitos"]
reais = classes[reservadas]
print("Erros corrigidos ao acrescentar as categorias:", int(np.sum((antes != reais) & (depois == reais))))
print("Novos erros ao acrescentar as categorias:", int(np.sum((antes == reais) & (depois != reais))))


def atributos_na_arvore(no, limite, nivel=0):
    if "atributo" not in no or (limite is not None and nivel >= limite):
        return set()
    return (
        {colunas[no["atributo"]]}
        | atributos_na_arvore(no["sim"], limite, nivel + 1)
        | atributos_na_arvore(no["nao"], limite, nivel + 1)
    )


print("Atributos usados nos testes da árvore com hábitos:")
print(sorted(atributos_na_arvore(arvores_habitos["Medidas e hábitos"], profundidade_escolhida)))

Nas 104 amostras reservadas
Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Quatro medidas          |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Matriz: [[43, 7], [9, 45]]
Medidas e hábitos       |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Matriz: [[43, 7], [9, 45]]
Erros corrigidos ao acrescentar as categorias: 0
Novos erros ao acrescentar as categorias: 0
Atributos usados nos testes da árvore com hábitos:
['GLUCOSE', 'HEMOGLOBIN', 'SYSTOLIC', 'TRIGLYCERIDES']


In [18]:
with np.load(caminho_registro, allow_pickle=False) as registro:
    dados = {chave: registro[chave].copy() for chave in registro.files}
dados["habitos_ordem"] = json.dumps(list(resultados_habitos))
dados["habitos_rodadas"] = np.array(list(resultados_habitos.values()))
dados["habitos_variantes"] = json.dumps(variantes_habitos)
dados["habitos_reserva"] = np.array(list(reservas_habitos.values()))
np.savez_compressed(caminho_registro, **dados)
print("Comparação registrada:", caminho_registro.relative_to(raiz))

Comparação registrada: resultados\arvore_busca_aulas.npz


#### O que encontramos

Disponibilizar tabagismo, atividade física e histórico familiar não mudou as métricas nas divisões comparadas. Nas 104 amostras reservadas, nenhuma previsão mudou.

A árvore ajustada com os sete atributos disponíveis usou apenas os quatro numéricos nos seus testes. Com a profundidade, o mínimo por folha e a estimativa de ausências mantidos, os testes categóricos não foram escolhidos pelo ganho de informação. A candidata de quatro atributos continua sendo a versão mais curta desta comparação.


### IMC e indicador binário acima do peso

Calculamos **IMC = peso em kg / altura em metros ao quadrado**. Para o indicador deste experimento, usamos **sim quando IMC é maior ou igual a 25** e **não quando é menor que 25**. O grupo sim inclui sobrepeso e obesidade.

O corte vem do [protocolo de sobrepeso e obesidade em adultos do Ministério da Saúde](https://www.gov.br/saude/pt-br/assuntos/pcdt/s/sobrepeso-e-obesidade-em-adultos), seção de diagnóstico. Criação de atributos e binarização são conteúdos da aula 2, páginas 82 a 91. A fórmula e o corte são referências para a variável da aplicação, e não uma regra fornecida pelo professor para a base.

A base contém idades variadas. Para este teste, o indicador matemático IMC maior ou igual a 25 é aplicado uniformemente. O ponto de corte citado é uma referência de adultos. O atributo não substitui uma avaliação nutricional específica por idade.

Mantemos a premissa de peso em quilogramas e altura já padronizada em centímetros. Se faltar peso ou altura, usamos a média daquele atributo no treinamento da rodada antes de calcular o IMC, seguindo o preenchimento da candidata. O resultado binário fica disponível para a árvore como mais uma pergunta de categoria.

Comparamos a árvore de quatro medidas, a árvore com o indicador de IMC e a árvore com IMC e os três atributos categóricos sugeridos. Mantemos profundidade, mínimo por folha e as mesmas divisões.

In [19]:
colunas_imc = colunas + ("IMC_GE_25",)
indice_imc = len(colunas)
habitos_imc = tuple(sorted(
    colunas.index(c) for c in ("SMOKER", "PHYSICAL_ACTIVITY", "FAMILY_HISTORY")
))
variantes_imc = {
    "Quatro medidas": tuple(atributos_escolhidos),
    "Medidas e IMC binário": tuple(atributos_escolhidos) + (indice_imc,),
    "Medidas, hábitos e IMC": tuple(atributos_escolhidos) + habitos_imc + (indice_imc,),
}


def preparar_com_imc(referencia, consulta):
    x, aval, parametros, mapas = preparar(referencia, consulta, preenchimento_escolhido)
    j_peso = colunas.index("WEIGHT")
    j_altura = colunas.index("HEIGHT")
    imc_ref = x[:, j_peso] / (x[:, j_altura] / 100) ** 2
    imc_aval = aval[:, j_peso] / (aval[:, j_altura] / 100) ** 2
    x = np.column_stack((x, (imc_ref >= 25).astype(int)))
    aval = np.column_stack((aval, (imc_aval >= 25).astype(int)))
    return x, aval


resultados_imc = {}
for semente in (42, 73, 101):
    rodadas = {nome: [] for nome in variantes_imc}
    for referencia, consulta in dividir(treinamento, semente):
        x, aval = preparar_com_imc(referencia, consulta)
        for nome, atributos in variantes_imc.items():
            arvore = construir_arvore(x, classes[referencia], atributos, folha_escolhida)
            previstas = np.array([prever(a, arvore, profundidade_escolhida) for a in aval])
            rodadas[nome].append(contar(classes[consulta], previstas))
    for nome, contagens in rodadas.items():
        resultados_imc[semente, nome] = np.array(contagens)

print("Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for nome in variantes_imc:
    combinado = np.concatenate((resultados_imc[42, nome], resultados_imc[73, nome]))
    mostrar(f"{nome}, s42/73", combinado)
for nome in variantes_imc:
    mostrar(f"{nome}, s101", resultados_imc[101, nome])

Avaliação               | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Quatro medidas, s42/73  |   90.39% |   90.37% |   90.51% |   91.50% |   89.36%
Medidas e IMC binário, s42/73 |   90.39% |   90.37% |   90.51% |   91.50% |   89.36%
Medidas, hábitos e IMC, s42/73 |   90.39% |   90.37% |   90.51% |   91.50% |   89.36%
Quatro medidas, s101    |   88.24% |   88.23% |   88.36% |   90.00% |   86.64%
Medidas e IMC binário, s101 |   88.24% |   88.23% |   88.36% |   90.00% |   86.64%
Medidas, hábitos e IMC, s101 |   88.24% |   88.23% |   88.36% |   90.00% |   86.64%


In [20]:
x_imc, aval_imc = preparar_com_imc(treinamento, reservadas)
reservas_imc = {}
arvores_imc = {}

print("Nas 104 amostras reservadas")
print("Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for nome, atributos in variantes_imc.items():
    arvore = construir_arvore(x_imc, classes[treinamento], atributos, folha_escolhida)
    previstas = np.array([prever(a, arvore, profundidade_escolhida) for a in aval_imc])
    contagens = contar(classes[reservadas], previstas)
    reservas_imc[nome] = contagens
    arvores_imc[nome] = arvore
    mostrar(nome, contagens)
    print("Matriz:", contagens.reshape(2, 2).tolist())


def atributos_usados_imc(no, limite, nivel=0):
    if "atributo" not in no or (limite is not None and nivel >= limite):
        return set()
    return (
        {colunas_imc[no["atributo"]]}
        | atributos_usados_imc(no["sim"], limite, nivel + 1)
        | atributos_usados_imc(no["nao"], limite, nivel + 1)
    )


print("Atributos usados na árvore com IMC binário:")
print(sorted(atributos_usados_imc(arvores_imc["Medidas e IMC binário"], profundidade_escolhida)))
print("Atributos usados na árvore com hábitos e IMC:")
print(sorted(atributos_usados_imc(arvores_imc["Medidas, hábitos e IMC"], profundidade_escolhida)))

Nas 104 amostras reservadas
Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Quatro medidas          |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Matriz: [[43, 7], [9, 45]]
Medidas e IMC binário   |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Matriz: [[43, 7], [9, 45]]
Medidas, hábitos e IMC  |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Matriz: [[43, 7], [9, 45]]
Atributos usados na árvore com IMC binário:
['GLUCOSE', 'HEMOGLOBIN', 'SYSTOLIC', 'TRIGLYCERIDES']
Atributos usados na árvore com hábitos e IMC:
['GLUCOSE', 'HEMOGLOBIN', 'SYSTOLIC', 'TRIGLYCERIDES']


In [21]:
with np.load(caminho_registro, allow_pickle=False) as registro:
    dados = {chave: registro[chave].copy() for chave in registro.files}
dados["imc_ordem"] = json.dumps(list(resultados_imc))
dados["imc_rodadas"] = np.array(list(resultados_imc.values()))
dados["imc_variantes"] = json.dumps(variantes_imc)
dados["imc_reserva"] = np.array(list(reservas_imc.values()))
dados["imc_corte"] = 25.0
np.savez_compressed(caminho_registro, **dados)
print("Comparação registrada:", caminho_registro.relative_to(raiz))

Comparação registrada: resultados\arvore_busca_aulas.npz


#### O que encontramos

Acrescentar o indicador IMC maior ou igual a 25 não mudou as métricas nas sementes 42, 73 e 101 nem nas 104 amostras reservadas. Acrescentar o indicador junto com tabagismo, atividade física e histórico familiar também manteve os resultados.

A árvore final, com os mesmos limites de tamanho, continuou escolhendo apenas pressão sistólica, glicose, hemoglobina e triglicerídeos. O novo indicador ficou disponível, mas não apareceu nas perguntas escolhidas pelo ganho de informação.


### Submissão da árvore de quatro atributos

Todos os testes anteriores estão registrados neste notebook, incluindo a grade inicial, a seleção dos quatro atributos, a conferência com outra semente, as categorias e o IMC binário.

Para a submissão, usamos a candidata de quatro medidas, porque os testes com tabagismo, atividade física, histórico familiar e IMC binário não mudaram os resultados. A configuração é pressão sistólica, glicose, hemoglobina e triglicerídeos, preenchimento pela média, mínimo de 15 exemplos por folha e profundidade máxima 6.

Os 90,39% de acurácia são a média das divisões usadas na seleção. A conferência com semente 101 ficou em 88,24%, e a reserva de 104 amostras ficou em 84,62%. A previsão no Kaggle é uma nova avaliação.

Para gerar o arquivo, recalculamos as médias e construímos a árvore com as 520 amostras rotuladas. Assim, os cortes podem mudar em relação à árvore impressa anteriormente, que foi ajustada nas 416 amostras.

In [22]:
with np.load(caminho_registro, allow_pickle=False) as registro:
    catalogo_original = json.loads(registro["catalogo"].item())
    configuracoes_registradas = json.loads(registro["selecao_configuracoes"].item())
    subconjuntos_registrados = json.loads(registro["selecao_subconjuntos"].item())
    posicao_envio = int(registro["selecao_escolhida"].item())
    configuracao_envio = configuracoes_registradas[posicao_envio]

if hashlib.sha256(caminho_base.read_bytes()).hexdigest() != catalogo_original["sha256_base"]:
    raise ValueError("A base mudou desde a execução da busca.")

grupo_envio, preenchimento_envio, folha_envio, profundidade_envio = configuracao_envio
indices_envio = tuple(subconjuntos_registrados[grupo_envio])
nomes_envio = tuple(colunas[j] for j in indices_envio)
if set(nomes_envio) != {"SYSTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"}:
    raise ValueError("A candidata registrada mudou. Conferir os atributos antes de enviar.")
if (preenchimento_envio, folha_envio, profundidade_envio) != ("media", 15, 6):
    raise ValueError("A configuração registrada mudou. Conferir antes de enviar.")

todos_rotulados = np.arange(len(amostras))
referencia_envio, _, medias_envio, mapas_envio = preparar(
    todos_rotulados, np.array([], dtype=int), preenchimento_envio
)
arvore_envio_completa = construir_arvore(
    referencia_envio, classes[todos_rotulados], indices_envio, folha_envio
)
arvore_envio = simplificar_arvore(arvore_envio_completa, profundidade_envio)

print("Amostras rotuladas na referência:", len(todos_rotulados))
print("Atributos:", ", ".join(nomes_envio))
print("Preenchimento:", preenchimento_envio)
print("Mínimo por folha:", folha_envio)
print("Profundidade máxima:", profundidade_envio)

Amostras rotuladas na referência: 520
Atributos: SYSTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Preenchimento: media
Mínimo por folha: 15
Profundidade máxima: 6


#### Preparação do arquivo de teste

O arquivo de teste ainda está no formato original. Aplicamos as regras numéricas da base padronizada, como altura em centímetros, pressão dividida em sistólica e diastólica e idades 0, -1 e 999 como ausentes.

Depois preenchemos as medidas ausentes com as médias calculadas nas 520 amostras rotuladas. A árvore consulta somente as quatro medidas escolhidas. Mantemos todas as linhas e os IDs na ordem original do teste.

In [23]:
def padronizar_teste_arvore(amostra):
    entrada = amostra.copy()
    if entrada["AGE"].strip() in {"0", "-1", "999"}:
        entrada["AGE"] = ""

    altura = ler_numero(entrada, "HEIGHT")
    if np.isfinite(altura) and altura < 3:
        entrada["HEIGHT"] = str(altura * 100)

    pressao = entrada["BLOOD_PRESSURE"].strip().replace("x", "/").replace(" ", "")
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"] = ""
        entrada["DIASTOLIC"] = ""
    else:
        sistolica, diastolica = pressao.split("/")
        entrada["SYSTOLIC"] = str(int(sistolica))
        entrada["DIASTOLIC"] = str(int(diastolica))
    return entrada


with (raiz / "dados" / "kaggle-test" / "test.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo_teste:
    teste_original = list(csv.DictReader(arquivo_teste))

ids_teste = [a["id"] for a in teste_original]
if not ids_teste or any(not identificador.strip() for identificador in ids_teste):
    raise ValueError("O teste precisa ter IDs preenchidos.")
if len(set(ids_teste)) != len(ids_teste):
    raise ValueError("Os IDs do teste devem ser únicos.")

entradas_teste = [padronizar_teste_arvore(a) for a in teste_original]
numeros_teste = np.array([
    [ler_numero(a, coluna) for coluna in numericas] for a in entradas_teste
])
for j, coluna in enumerate(numericas):
    numeros_teste[:, j] = np.where(
        np.isfinite(numeros_teste[:, j]), numeros_teste[:, j], medias_envio[coluna]
    )

previsoes_envio = np.array([prever(a, arvore_envio, None) for a in numeros_teste])
if not np.all(np.isin(previsoes_envio, [0, 1])):
    raise ValueError("As previsões devem pertencer às classes 0 e 1.")
print("Amostras no teste:", len(teste_original))
print("Previsões geradas:", len(previsoes_envio))

Amostras no teste: 312
Previsões geradas: 312


#### CSV para enviar ao Kaggle

Salvamos as colunas id e class, com uma previsão por linha do teste. O nome do arquivo identifica a árvore de quatro atributos.

No formulário da competição, selecionamos esse CSV. Uma descrição possível é “Árvore, 4 atributos, ganho de informação, média, folha 15, profundidade 6”. A pontuação recebida deve ser registrada separadamente das métricas locais.

In [24]:
caminho_envio_arvore = raiz / "resultados" / "submissao_arvore_4_atributos.csv"
linhas_envio = list(zip(ids_teste, previsoes_envio.tolist(), strict=True))

with caminho_envio_arvore.open("w", encoding="utf-8", newline="") as arquivo_envio:
    escritor = csv.writer(arquivo_envio)
    escritor.writerow(["id", "class"])
    escritor.writerows(linhas_envio)

with caminho_envio_arvore.open(encoding="utf-8", newline="") as arquivo_conferencia:
    leitor = csv.DictReader(arquivo_conferencia)
    if leitor.fieldnames != ["id", "class"]:
        raise ValueError("Cabeçalho incorreto no CSV.")
    linhas_conferencia = list(leitor)

if [a["id"] for a in linhas_conferencia] != ids_teste:
    raise ValueError("Os IDs do CSV não correspondem à ordem do teste.")
if any(a["class"] not in {"0", "1"} for a in linhas_conferencia):
    raise ValueError("Há uma previsão inválida no CSV.")

print("Arquivo salvo:", caminho_envio_arvore.relative_to(raiz))
print("Linhas de previsão:", len(linhas_conferencia))
print("SHA-256 do CSV:", hashlib.sha256(caminho_envio_arvore.read_bytes()).hexdigest())

Arquivo salvo: resultados\submissao_arvore_4_atributos.csv
Linhas de previsão: 312
SHA-256 do CSV: 5cb947fded97aa32cc7a80e5325cc2f2baf5b8e792b61968a8760a7b60b02f02


### Resultado no Kaggle

A submissão da árvore de quatro atributos foi processada com pontuação pública de **0,83018**, conforme resultado apresentado pelo aluno em 05/11/2026. Essa pontuação é registrada separadamente da acurácia de 90,39% nas divisões da seleção e de 84,62% nas amostras reservadas.

A comparação da árvore com o primeiro KNN K=11 e a regra de combinação estão em [01_Knn_arvore.ipynb](../Mod5_KNN_AD/01_Knn_arvore.ipynb).


### Referências

- [Aula 2](../docs/materiais_aula/aula2_MD.pdf), páginas 44 a 47, 52 e 78 a 91.
- [Aula 4](../docs/materiais_aula/aula4_MD.pdf), página 9.
- [Aula 5](../docs/materiais_aula/aula5_MD.pdf), páginas 16 a 28.
- [Aula 7](../docs/materiais_aula/aula7_MD.pdf), páginas 7 a 11.

As páginas são contadas a partir da capa do PDF. A varredura de cortes e os limites de tamanho da árvore são escolhas da implementação que aplica os conceitos de binarização, ganho de informação e árvores compactas. As premissas sobre as medidas da base continuam registradas como decisões do trabalho.